<img src="../../images/SnowparkIconLabel.png" alt="Snowpark Icon" width=150px align=right /> 

# Stored Procedures and Snowpark

The Snowpark Python API provides mechanisms for creating, registering, and calling stored procedures programmatically. Like UDFs, stored procedures can be temporary or permanent. If temporary, the procedure will only be available in the current session and will be automatically dropped at the end of the session. Any procedure can be manually dropped as long as the role has ownership of the procedure.

The Snowpark Python API has a package for stored procedures named `snowflake.snowpark.stored_procedure`.

Additionally, the object `snowflake.snowpark.functions` has a method named `sproc(...)` that registers a Python function as a Snowflake Python stored procedure and returns the stored procedure. Be careful not to confuse this function with the property `snowflake.snowpark.Session.sproc`. This property is an instance of `snowflake.snowpark.stored_procedure.StoredProcedureRegistration`. 

The `Session` class contains a `.call(..)` function for invoking stored procedures. It returns an `Any` and not a `DataFrame`. 

> **&#128221; Note:** See documentation for further details:
> - [Snowpark Stored Procedures Home](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/stored_procedures.html#stored-procedures)
> - [snowflake.snowpark.stored_procedure.StoredProcedure](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.stored_procedure.StoredProcedure.html#snowflake-snowpark-stored-procedure-storedprocedure)
> - [snowflake.snowpark.stored_procedure.StoredProcedureRegistration](https://docs.snowflake.com/en/developer1.guide/snowpark/reference/python/api/snowflake.snowpark.stored_procedure.StoredProcedureRegistration.html#snowflake1.snowpark1.stored-procedure-storedprocedureregistration)
> - [Python typing.Any](https://docs.python.org/3/library/typing.html?highlight=any#typing.Any)


## UDF vs Stored Procedure
<img src="../../images/UDFvSproc.png" alt="UDF_Sproc_table" style="width:85%;display:block;margin:5%" /> 
<br />

> **&#128221; Note:** See documentation for further details:
> - [Writing Files](https://docs.snowflake.com/en/developer-guide/udf/python/udf-python-examples#label-udf-python-write-files)
> - [Unzipping a Stage File](https://docs.snowflake.com/en/developer-guide/udf/python/udf-python-examples#label-udf-python-unzipping-file-from-stage)

Stored procedures can be overloaded and follow the [general guidelines for objects identifiers](https://docs.snowflake.com/sql-reference/identifiers) in Snowflake along with some object-specific restrictions and "gotchas".

> **&#128221; Note:** See documentation for further details:
> - [Naming and Overloading Procedures and UDFs](https://docs.snowflake.com/developer-guide/udf-stored-procedure-naming-conventions)

## Topics in this Lesson

1. [Calling a Stored Procedure](#Calling_a_Stored_Procedure)    
    1. [Using SQL (`Session.sql(...)`)](#Calling_a_Stored_Procedure_Using_SQL)    
    1. [Programmatically Using `Session.call(...)`](#Programmatically_Using_Session_call)  
    1. [Programmatically Using `StoredProcedure(..)`](#Programmatically_Using_StoredProcedure)  
1. [Authoring a Python Stored Procedure Using a Python Function](#Authoring_a_Python_Stored_Procedure)  
    1. [Lambda Example](#SP_lambda_example)  
    1. [Named Example](#SP_named_example)  
1. [Using `sproc(...)` as a Decorator](#Using_sproc_decorator)  
    1. [Anonymous, Named, Call (With Anonymous) Recap](#anon_named_call_with_recap)  
    1. [Interacting with Your Snowflake Account from within a Stored Procedure](#interaction_with_your_sf_account)  
    1. [Table Procedures a.k.a. Tabular Procedures (Preview)](#table_procedures)     
    1. [The `RETURNS` Clause in SQL `CREATE PROCEDURE` Statements](#The_RETURNS_Clause_in_SQL_CREATE_PROCEDURE_Statements)   
    1. [Using Custom and Third Party Libraries](#Using_Libraries)  
1. [Clean Up and Close Session](#AE7_cleanup)
1. [Lab: Instructor Demo - Creating Stored Procedures in Snowsight Using Python Worksheets (Preview)](#lab_tbd)  
1. [Additional Examples](#Sprocs_Additional_Examples)  

#### Connect and create a `Session`
1. Import required libraries.  
1. Create a `Session` to connect to Snowflake.  
<br />  
    > &#10071; Success requires that you have already completed the key pair authentication exercise.  
1. Set context items for this module.  

Run the following cell to connect to your Snowflake account. 
<br />
*You needn't edit anything in the following cell. Just run it.*

In [1]:
# Run utils notebook
%run ../../utils/ds_utils_python.ipynb

# Connect to Snowflake and create a Session object named session
session = create_session()

Read of key/pair authentication objects successful
User authenticated and session created
----------------------------------------------------------
|"Session Property Name"      |"Session Property Value"  |
----------------------------------------------------------
|version                      |1.6.1                     |
|python.version               |3.8.18                    |
|python.connector.version     |3.7.0                     |
|python.connector.session.id  |396816324056606           |
|os.name                      |Linux                     |
----------------------------------------------------------



#### Setup Code

We should set ourselves up for success. The following ensures our context is set properly for database, schema, role, and warehouse.

*You needn't edit anything in the following cell. Just run it.*

In [2]:
# Hard code the lesson name
lesson_name = "STORED_PROCEDURES"

# Create the context items for this lesson
lesson = confirm_or_create_lesson_context(session, lesson_name)

Creation of context items could take a few moments... be patient
The current user is: RAVEN
Query tag is: Data Science: RAVEN - STORED_PROCEDURES
-------------------------------------------------
|"Created warehouse RAVEN_WH"                   |
-------------------------------------------------
|RAVEN_WH already exists, statement succeeded.  |
-------------------------------------------------

------------------------------------
|"Altered warehouse RAVEN_WH"      |
------------------------------------
|Statement executed successfully.  |
------------------------------------

Setting current warehouse to RAVEN_WH
Creating database RAVEN_DB
-------------------------------------------------
|"Created database RAVEN_DB"                    |
-------------------------------------------------
|RAVEN_DB already exists, statement succeeded.  |
-------------------------------------------------

Creating schema STORED_PROCEDURES_LESSON
---------------------------------------------------------
|"

<a id="Calling_a_Stored_Procedure"></a>
## 1. Calling a Stored Procedure 

<a id="Calling_a_Stored_Procedure_Using_SQL"></a>
#### 1A. Using SQL (`Session.sql(...)`)

Calling a stored procedure can be accomplished by using the `Session.sql(...)` method.

```python
sp_ret_value_df = session.sql("CALL MY_STORED_PROCEDURE('a',7,true,-40.0)") # Creates a DataFrame
    
sp_ret_value = sp_ret_value_df.collect()[0][0] # Action to retrieve a row/column value
```

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.Session.sql(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/api/snowflake.snowpark.Session.sql)
> - [Snowflake Snowpark Docs - Calling Functions and Stored Procedures in Snowpark Python](https://docs.snowflake.com/en/developer-guide/snowpark/python/calling-functions.html#calling-functions-and-stored-procedures-in-snowpark-python)

<a id="Programmatically_Using_Session_call"></a>
#### 1B. Programmatically Using `Session.call(...)`

Calling a stored procedure can be accomplished by using the `Session.call(...)` method.

```python
sp_ret_value = (
    session.call(
        sproc_name = "MY_STORED_PROCEDURE" # Stored procedure name
       ,"a"    # First arg to stored procedure
       ,7      # Second arg to stored procedure
       ,True   # Third arg to stored procedure
       ,-40    # Fourth arg to stored procedure
    )
)
```

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.Session.call(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.call.html#snowflake.snowpark.Session.call)
> - [Snowflake Snowpark Docs - Calling Functions and Stored Procedures in Snowpark Python](https://docs.snowflake.com/en/developer-guide/snowpark/python/calling-functions.html#calling-functions-and-stored-procedures-in-snowpark-python)

<a id="Programmatically_Using_StoredProcedure"></a>
#### 1C. Programmatically Using `StoredProcedure()`

When you register a stored procedure with the sproc function, it returns a func object. You can then call this object like a regular Python function, using parentheses.

```python
from snowflake.snowpark.functions import sproc
my_sp_object = (
    sproc(
        ...
        input_types = [<parameter types here>,...,...]
    )
)
# Call the Stored Procedure with arguments
return_value = my_sp_object(<args here>,...,...)
```

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.stored_procedure.StoredProcedure.func](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/api/snowflake.snowpark.stored_procedure.StoredProcedure.func)
> - [Snowflake Snowpark Docs - Calling Functions and Stored Procedures in Snowpark Python](https://docs.snowflake.com/en/developer-guide/snowpark/python/calling-functions.html#calling-functions-and-stored-procedures-in-snowpark-python)

<a id="Authoring_a_Python_Stored_Procedure"></a>

## 2. Authoring a Python Stored Procedure Using a Python Function
    
Unlike the Snowpark APIs for Java and Scala, the Snowpark Python API has types, classes, and objects available to support developing your stored procedures in Python code *and* that code can be created on a remote client like a notebook or an IDE.

An anonymous lambda function or a named function can be used as the power to a stored procedure. For this you will need to use `snowflake.snowpark.functions.sproc(...)`.  Additionally, a named function could be passed to `sproc(...)`.  This function returns a `StoredProcedure` function that can be called directly.

This function takes the following arguments:

- **func** – A Python function used for creating the stored procedure. Can be a lambda function or a named function.
- **return_type** – A `DataType` representing the return data type of the stored procedure.
- **input_types** – A `list` of `DataType`s representing the input datatypes of the stored procedure.
- **name** – A string or `list` of strings that specify the name or fully-qualified object identifier (database name, schema name, and function name) for the stored procedure in Snowflake, which allows you to call this stored procedure in a SQL command or via session.call(). If it is not provided, a name will be automatically generated for the stored procedure. A name must be specified when `is_permanent` is `True`.
- **is_permanent** – Whether to create a permanent stored procedure. The default is `False`. If it is `True`, a valid **stage_location** must be provided.
- **stage_location** – The stage location where the Python file for the stored procedure and its dependencies should be uploaded. The stage location must be specified when **is_permanent** is `True`, and it will be ignored when **is_permanent** is `False`. It can be any stage other than temporary stages and external stages.
- **imports** – A `list` of imports that only apply to this stored procedure. You can use a string to represent a file path (similar to the path argument in `add_import()`) in this `list`, or a `tuple` of two strings to represent a file path and an `import` path (similar to the `import_path` argument in `add_import()`). These stored-proc-level imports will override the session-level imports added by `add_import()`.
- **packages** – A `list` of packages that only apply to this stored procedure. These stored-proc-level packages will override the session-level packages added by `add_packages()` and `add_requirements()`.
- **replace** – Whether to replace a stored procedure that already was registered. The default is `False`. If it is `False`, attempting to register a stored procedure with a name that already exists results in a `SnowparkSQLException` exception being thrown. If it is `True`, an existing stored procedure with the same name is overwritten.
- **session** – Use this session to register the stored procedure. If it’s not specified, the session that you created before calling this function will be used. You need to specify this parameter if you have created multiple sessions before calling this method. <span style="color:red">For this notebook, if you re-run the session setup cell, you will have multiple session objects in scope and an error will occur telling you this for any invocation where `session` is not specified. To clear all `Session` instances, shut down and restart your kernel. Then run both setup cells.</span> 
- **parallel** – The number of threads to use for uploading stored procedure files with the `PUT` command. The default value is 4 and supported values are from 1 to 99. Increasing the number of threads can improve performance when uploading large stored procedure files.
- **statement_params** – Dictionary of statement level parameters to be set while executing this action. (e.g. "query_tag")

> **&#128221; Note:** See documentation for further details:
> - [Snowpark Stored Procedures Home](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/stored_procedures.html)
> - [snowflake.snowpark.functions.sproc(...)](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.functions.sproc.html#snowflake.snowpark.functions.sproc)
> - [snowflake.snowpark.Session.sproc](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.Session.sproc.html#snowflake.snowpark.Session.sproc) (property of type [StoredProcedureRegistration](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/api/snowflake.snowpark.stored_procedure.StoredProcedureRegistration.html#snowflake.snowpark.stored_procedure.StoredProcedureRegistration))


<a id="SP_lambda_example"></a>

### 2A. Lambda Example
Let's see a simple stored procedure example using a lambda function and then we will use a named function in the same way. This procedure will simply add two numeric arguments (doubles) together and return the result (double). 

In [3]:
# Import needed types and objects
from snowflake.snowpark.functions import sproc
from snowflake.snowpark.types import DoubleType

# Create a StoredProcedure object that uses a lambda 
my_sp_lambda = (
    sproc(
            # The first argument is a simple lambda function object that adds two arguments together
         func = lambda session, x, y: x + y  # The lambda function takes 3 arguments: session, x, and y
                                             # The body of the function is on the right of the colon
                                             # The body adds x to y and returns that value ("return" is understood)
        ,return_type = DoubleType()                # What does the SP return?
        ,input_types = [DoubleType(),DoubleType()] # What are the datatypes of the arguments to the SP?
        ,is_permanent = False                      # If false (the default), the procedure won't outlive the session (temporary procedure)              
        ,packages=["snowflake-snowpark-python"]    # What packages does this SP need for runtime success?
        ,replace = True                            # Replace if already exists. (Set to true in case we want to run this cell again.)
        ,session = session                         # Our previously created session will be used to register the stored procedure.
   )    
)

print(f"\nmy_sp_lambda object created and is of type {type(my_sp_lambda)}")
print(f"It was dynamically named {my_sp_lambda.name} which can be used in a SQL statement")
print("\nCalling my_sp_lambda(...)")
ret_value_lambda = my_sp_lambda(3.0, 5.0)
print(f"ret_value_lambda = {ret_value_lambda}")


my_sp_lambda object created and is of type <class 'snowflake.snowpark.stored_procedure.StoredProcedure'>
It was dynamically named "RAVEN_DB"."STORED_PROCEDURES_LESSON".SNOWPARK_TEMP_PROCEDURE_7YH4XMVJS6 which can be used in a SQL statement

Calling my_sp_lambda(...)
ret_value_lambda = 8.0


---
We can show the procedures using SQL to confirm their creation.

In [4]:
# Import needed types, objects, and functions
from snowflake.snowpark.functions import col, split, lit
from snowflake.snowpark.types import StringType

# Create a column just for the return type for verbosity
return_type_column = split(col('"arguments"'),lit(" RETURN "))[1].cast(StringType())

# Invoke SHOW USER PROCEDURES via session.sql(...)_
show_procedures_df = (session.sql(f"SHOW USER PROCEDURES IN SCHEMA")
     .select('"name"','"arguments"')
     .with_column("RETURN_TYPE", return_type_column)
)
show_procedures_df.show()

-----------------------------------------------------------------------------------------------------------
|"name"                              |"arguments"                                         |"RETURN_TYPE"  |
-----------------------------------------------------------------------------------------------------------
|SNOWPARK_TEMP_PROCEDURE_7YH4XMVJS6  |SNOWPARK_TEMP_PROCEDURE_7YH4XMVJS6(FLOAT, FLOAT...  |FLOAT          |
-----------------------------------------------------------------------------------------------------------



<a id="SP_named_example"></a>

### 2B. Named Example

The previous example was a stored procedure powered by a lambda (anonymous) function. The below shows how the same functionality could be created using a named Python function instead of a lambda function. 

In [5]:
# Import needed types, objects, and functions
from snowflake.snowpark.functions import sproc
from snowflake.snowpark.types import DoubleType

# Define a named Python function
# The first argument must be a Session instance
def my_add(session:Session, x, y) -> float:  # returns a float
    return x + y

print(f"Created Python function my_add(Session,x,y)")

# Create a StoredProcedure object that uses a named Python function 
my_sp_named = (sproc(
     func = my_add                             # The named Python function
    ,return_type = DoubleType()                # The return type of the procedure
    ,input_types = [DoubleType(),DoubleType()] # The types for the arguments to the procedure
    ,name = "ADD_USING_NAMED"                  # The name to be used in SQL CALL statements
    ,packages = ["snowflake-snowpark-python"]  # Package dependencies
    ,session = session
    ,replace = True                            # Replace if already exists 
   )                                           #   (in case we want to run this cell multiple times)       
)

print(f"Create procedure my_sp_named of type {type(my_sp_named)}")
print("\nThree invocations:")
print("\nUsing session.call(...)")
ret_value1 = session.call("ADD_USING_NAMED",3.0,5.0)
print(f"ret_value1 = {ret_value1}")

print("\nUsing session.sql(...).show()")
ret_df = session.sql("CALL ADD_USING_NAMED(3,5)").show()

print("Using my_sp_named(...)")
ret_value2 = my_sp_named(3,5)
print(f"ret_value2 = {ret_value2}")

print("Let's see all procedures")
show_procedures_df.show()

Created Python function my_add(Session,x,y)
Create procedure my_sp_named of type <class 'snowflake.snowpark.stored_procedure.StoredProcedure'>

Three invocations:

Using session.call(...)
ret_value1 = 8.0

Using session.sql(...).show()
---------------------
|"ADD_USING_NAMED"  |
---------------------
|8.0                |
---------------------

Using my_sp_named(...)
ret_value2 = 8.0
Let's see all procedures
-----------------------------------------------------------------------------------------------------------
|"name"                              |"arguments"                                         |"RETURN_TYPE"  |
-----------------------------------------------------------------------------------------------------------
|ADD_USING_NAMED                     |ADD_USING_NAMED(FLOAT, FLOAT) RETURN FLOAT          |FLOAT          |
|SNOWPARK_TEMP_PROCEDURE_7YH4XMVJS6  |SNOWPARK_TEMP_PROCEDURE_7YH4XMVJS6(FLOAT, FLOAT...  |FLOAT          |
------------------------------------------------

--- 

Let's view the DDL for our procedure `ADD_USING_NAMED(FLOAT,FLOAT)`. 

In [6]:
from snowflake.snowpark.functions import call_builtin
(session.create_dataframe([""])
    .select(
            call_builtin(
                 "GET_DDL"
                ,"PROCEDURE"
                ,"ADD_USING_NAMED(FLOAT,FLOAT)"
        )
    )
    .show(1,100)
)

--------------------------------------------------------------------------------------------------------
|"GET_DDL('PROCEDURE', 'ADD_USING_NAMED(FLOAT,FLOAT)')"                                                |
--------------------------------------------------------------------------------------------------------
|CREATE OR REPLACE PROCEDURE "ADD_USING_NAMED"("ARG1" FLOAT, "ARG2" FLOAT)                             |
|RETURNS FLOAT                                                                                         |
|LANGUAGE PYTHON                                                                                       |
|RUNTIME_VERSION = '3.8'                                                                               |
|PACKAGES = ('snowflake-snowpark-python','cloudpickle==2.0.0')                                         |
|HANDLER = 'compute'                                                                                   |
|EXECUTE AS OWNER                                      

--- 

<span style="color:red;">Important:</span> Notice the DDL above. Our procedure is **not** declared as `TEMPORARY` (or `TEMP`). That wouldn't be valid syntax. However, the procedures we created so far are both "temporary". They will not outlive this session. If you create another Snowflake session and `SHOW USER PROCEDURES` you will not see either one. 

When we created our two procedures, `ADD_USING_LAMBDA(FLOAT,FLOAT)` and `ADD_USING_NAMED(FLOAT,FLOAT)`, we did not use `True` for the argument `is_permanent` (the default for this parameter is `False`). Had we passed `True` **and** supplied a `stage_location` as an additional argument, the procedures would be permanent. We can use a permanent procedure long after our Snowpark session has ended until someone drops the procedure. The same can be said for temporary vs permanent UDFs and UDTFs. See the lesson titled [*Creating and Registering User-Defined Functions (UDFs)*](./06-Creating-and-Registering-UDFs.ipynb) for more on how to make functions permanent. The same logic applies for making stored procedures permanent.  

<a id="Using_sproc_decorator"></a>

## 3. Using `sproc(...)` as a Decorator

We saw how the method `sproc(...)` can create and register a stored procedure from a lambda or named function.  The `sproc(...)` function can be used as a decorator on a Python function as well. 

```python
from snowflake.snowpark import Session
from snowflake.snowpark.functions import sproc

@sproc(name = "<the name>", input_types = [<list of DataType objects>], return_type = <DataType>, etc...) 
def myFunc(session:Session, <other parameters>):
    <body>
```

**Note:** This could also be accomplished using `StoredProcedureRegistration.register(...)`. As usual there are multiple ways of doing the same thing in Snowpark. 

The two functions....

```python
my_sp_1 = functions.sproc(...)

```

and

```python
my_sp_2 = session.sproc.register(...) # StoredProcedureRegistration.register(...)

```

... take the same arguments and return the same thing, a `StoredProcedure` object. 

> **&#128221; Note:** See documentation for further details:
> - [snowflake.snowpark.stored_procedure.StoredProcedureRegistration.register(....)](https://docs.snowflake.com/developer-guide/snowpark/reference/python/latest/api/snowflake.snowpark.stored_procedure.StoredProcedureRegistration.register)
> - [snowflake.snowpark.functions.sproc(...)](https://docs.snowflake.com/developer-guide/snowpark/reference/python/latest/api/snowflake.snowpark.functions.sproc)
> - [snowflake.snowpark.stored_procedure.StoredProcedure](https://docs.snowflake.com/developer-guide/snowpark/reference/python/latest/api/snowflake.snowpark.stored_procedure.StoredProcedure#snowflake.snowpark.stored_procedure.StoredProcedure)



For the example below, we are creating a stage to use for the parameter `stage_location`. By specifying a `stage_location` and passing `True` for `is_permanent`, we are making the procedure permanent. 

As of Snowpark `1.12.1`, the `Session` class contains no function for reflecting on the current user. The stored procedure below provides us with a mechanism for discovering the current user's identity programmatically. 

1. Create a stage if the procedure is to be permanent.
1. Import `snowflake.snowpark.functions.sproc`.
1. Define a Python function taking a `Session` object as its first argument.
1. Decorate the Python function using the `@sproc(...)` annotation.

Then, of course, you invoke the stored procedure using `session.sql("CALL....").<action>` or `session.call(...)`. 

In [7]:
# Import needed types, objects, and functions
from snowflake.snowpark import Session
from snowflake.snowpark.functions import sproc
from snowflake.snowpark.types import StringType

# Create a stage for the bytecode of the stored procedure
stage_name = f"{get_lesson().full_schema()}.SP_STAGE_DEMO"
(session.sql(f"CREATE OR REPLACE STAGE {stage_name}")
  .show()
)
print(f"Created stage {stage_name}")
print(f"\nCreating an annotated Python function get_current_username with the procedure name GET_USERNAME_SP.")
print(f"\nThis might take a second or two.")

@sproc( name="GET_USERNAME_SP"
       ,return_type = StringType()
       ,input_types = []
       ,is_permanent = True
       ,stage_location = f"@{stage_name}"
       ,replace = True
       ,packages = ["snowflake-snowpark-python"]
       ,session = session)
def get_current_username(session:Session) -> str:
    return session.sql("SELECT CURRENT_USER()").collect()[0][0]

print('\nInvoking the SP using SQL (session.sql("CALL GET_USERNAME_SP()"))')
df_val = session.sql("CALL GET_USERNAME_SP()").collect()[0][0]
print(f"Return value is: {df_val}")

print('\nInvoking the SP programatically (session.call("GET_USERNAME_SP"))')
ret_val = session.call("GET_USERNAME_SP")
print(f"Return value is: {ret_val}")

print("\nInvoking the SP programatically (get_current_username())")
print("\nThe passed session object is the client-side session object")
ret_val = get_current_username()
print(f"Return value is: {ret_val}")

print("Show all user procedures")
show_procedures_df.show(20)

--------------------------------------------------
|"status"                                        |
--------------------------------------------------
|Stage area SP_STAGE_DEMO successfully created.  |
--------------------------------------------------

Created stage RAVEN_DB.STORED_PROCEDURES_LESSON.SP_STAGE_DEMO

Creating an annotated Python function get_current_username with the procedure name GET_USERNAME_SP.

This might take a second or two.

Invoking the SP using SQL (session.sql("CALL GET_USERNAME_SP()"))
Return value is: RAVEN

Invoking the SP programatically (session.call("GET_USERNAME_SP"))
Return value is: RAVEN

Invoking the SP programatically (get_current_username())

The passed session object is the client-side session object
Return value is: RAVEN
Show all user procedures
-----------------------------------------------------------------------------------------------------------
|"name"                              |"arguments"                                         |"

---

<a id="anon_named_call_with_recap"></a>
### 3A. Anonymous, Named, Call (With Anonymous) Recap

<img src="../../images/ProcedureTypes.png" alt="Sproc Types" width=65% />
<br />

<sup>*</sup> The owning role of a schema does not need any explicit privileges to create procedures in that schema. 

> **&#128221; Note:** "Call (with Anonymous Procedure)" is a Snowflake SQL idiom that allows for procedural logic to be invoked without requiring the the calling role to have the privilege of creating a Stored Procedure object in the current database & schema. You can find more on this in the section [*CALL (with Anonymous Procedure)*](../Appendices/Appendix-Additional-Lecture-Examples.ipynb#AE_call_with_anon_sproc) in the *Additiona Lecture Examples* appendix. Additionally, see the Snowflake SQL documentation as well.
> - Snowflake SQL Documentation: [CALL (with Anonymous Procedure)](https://docs.snowflake.com/en/sql-reference/sql/call-with)

--- 

<a id="interaction_with_your_sf_account"></a>
### 3B. Interacting with Your Snowflake Account from within a Stored Procedure

When designing a stored procedure (SP) using Python, the first argument to your handler method will (must!) be an object of type `Session`. When your handler function is invoked as a stored procedure, it won't execute within your remote client application. It will execute within a single node of your Snowflake virtual warehouse.

You will not create (nor provide) the `Session` instance that is passed to your stored procedure handler method. The Snowflake platform will create a `Session` object and pass it to your handler method along with any additional arguments the user provides. 

The `Session` object you created earlier in this notebook resides on the client (this Jupyter notebook and kernel). It will **not** (and cannot) be used within any stored procedure handler function. 

In [8]:
print(f"The Session object created for this lesson: {session}")
print("It's a client-side Session instance")

The Session object created for this lesson: <snowflake.snowpark.session.Session: account="QXB43139", role="TRAINING_ROLE", database="RAVEN_DB", schema="STORED_PROCEDURES_LESSON", warehouse="RAVEN_WH">
It's a client-side Session instance


---

The `Session` object that will be passed to your handler method of your SP will reside on the "server side". 

```python
from snowflake.snowpark import Session, DataFrame

@sproc(name="EXAMPLE_SP", return_type = ..., input_types = [<arg types>], is_permanent=<boolean>, stage_location="@{<stage_name>}", replace=<boolean>, packages=["snowflake-snowpark-python"])
def do_something(session:Session, arg1, arg2 ) -> <return type>:
       # The Session instance ^^ here does not reside on any remote client. 
       # It is created and resides WITHIN a server node of your Snowflake virtual warehouse
       # You can use it to interact with your Snowflake account
    myDF1 = session.sql("<arbitrary sql>")
    myDF2 = session.table("<a table or view name>")
    joinedDF = myDF1.join(myDF2,...)
    resultsDF = joinedDF.select(...) # group? agg? sort? filter? select?
    
    row_array = resultsDF.collect()       # Retrieve data if desired
    for row in row_array:
        save_the_world(row)
       
    resultsDF.write.save_as_table(....)   # Write the results if desired
    resultsDF.create_or_replace_view(...) # Create a view perhaps
```

In [9]:
# Typical imports
from snowflake.snowpark.types import StringType
from snowflake.snowpark.functions import sproc
from snowflake.snowpark import Session

# Here we "decorate" the get_session_info python function with sproc as an annotation
@sproc( name="GET_SERVER_SIDE_SESSION_INFO"
       ,return_type = StringType()
       ,input_types = []
       ,is_permanent= True
       ,stage_location= f"@{stage_name}"
       ,replace= True
       ,packages=["snowflake-snowpark-python"])
def get_session_info(server_side_session:Session) -> str:
    return server_side_session._session_info

print(f"Permanent stored procedure named GET_SERVER_SIDE_SESSION_INFO created.")
print("This may take a few seconds.")

Permanent stored procedure named GET_SERVER_SIDE_SESSION_INFO created.
This may take a few seconds.


---

Below, we echo the session information for the `Session` object we created at the top of this notebook. This `Session` object resides on the remote client. 

In [10]:
print(f"The info for this JupyterLabs notebook Snowpark Session object (client-side) is:\n{session._session_info}")

The info for this JupyterLabs notebook Snowpark Session object (client-side) is:

"version" : 1.6.1,
"python.version" : 3.8.18,
"python.connector.version" : 3.7.0,
"python.connector.session.id" : 396816324056606,
"os.name" : Linux



---
Below we are invoking the stored procedure we named "GET_SESSION_INFO". The handler code for the procedure is passed a server-side `Session` object created by the Snowflake platform. The versions on the client-side could differ from those on the server-side. 

In [11]:
sp_return_value = session.call("GET_SERVER_SIDE_SESSION_INFO")
print(f"The info for the Session object in an instance of our Stored Procedure running within our Snowflake account (server-side) is:\n{sp_return_value}")

The info for the Session object in an instance of our Stored Procedure running within our Snowflake account (server-side) is:

"version" : 1.22.1,
"python.version" : 3.8.19,
"python.connector.version" : 0.31.0,
"python.connector.session.id" : None,
"os.name" : Linux



---

The goal of the below procedure is to interact with the Snowflake account using the`Session` object that is provided by the Snowflake account on the server-side at runtime. This `Session` object is passed as the first argument to any handler function of a stored procedure. Don't confuse this object with the `Session` that is created on the client-side for interaction from a remote Snowpark application. 

In [12]:
# Import needed types, functions, and objects
from snowflake.snowpark.functions import builtin, uniform, lit, seq1
from snowflake.snowpark.types import StringType, IntegerType, BooleanType
import random

print(f"The stage for holding the permanent stored procedure is @{stage_name}")

@sproc( name="CREATE_AND_POP_TABLE_SP"
       ,return_type = StringType()
       ,input_types = [StringType(), IntegerType(), BooleanType()]
       ,is_permanent= True
       ,stage_location= f"@{stage_name}"
       ,replace= True
       ,packages=["snowflake-snowpark-python"])
def create_and_populate_table(sess:Session, receiving_table_name, row_count_to_generate, overwrite:bool) -> str:
    # The session object used here (sess) is not this notebook's session that we created at the begining of this lesson.
    df = (
         sess.generator(  
            seq1().alias("Sequence")
           ,uniform(50, 80,col("Sequence")).alias("Uniform Values 50-80")
           ,rowcount = row_count_to_generate
          )
       )
    ret = f"{row_count_to_generate} rows appended to table {receiving_table_name}"
    the_mode = "append"
    
    # Caller of the procedure decides if rows will be appended or overwritten
    if overwrite:
        ret = f"{row_count_to_generate} rows written to table {receiving_table_name}"
        the_mode = "overwrite"
    try:        
        (df.write.save_as_table(
             table_name = receiving_table_name # Table name passed into this procedure
            ,mode = the_mode                   # Either append or overwrite
        )
    )
    except Exception as e:
        ret = ret + f"\tError:{str(e)}"
    return ret

show_procedures_df.show()

The stage for holding the permanent stored procedure is @RAVEN_DB.STORED_PROCEDURES_LESSON.SP_STAGE_DEMO
-----------------------------------------------------------------------------------------------------------
|"name"                              |"arguments"                                         |"RETURN_TYPE"  |
-----------------------------------------------------------------------------------------------------------
|ADD_USING_NAMED                     |ADD_USING_NAMED(FLOAT, FLOAT) RETURN FLOAT          |FLOAT          |
|CREATE_AND_POP_TABLE_SP             |CREATE_AND_POP_TABLE_SP(VARCHAR, NUMBER, BOOLEA...  |VARCHAR        |
|GET_SERVER_SIDE_SESSION_INFO        |GET_SERVER_SIDE_SESSION_INFO() RETURN VARCHAR       |VARCHAR        |
|GET_USERNAME_SP                     |GET_USERNAME_SP() RETURN VARCHAR                    |VARCHAR        |
|SNOWPARK_TEMP_PROCEDURE_7YH4XMVJS6  |SNOWPARK_TEMP_PROCEDURE_7YH4XMVJS6(FLOAT, FLOAT...  |FLOAT          |
-------------------------------

---

The cell below invokes our stored procedure with different values and echoes that the table information has changed. 

In [14]:
from snowflake.snowpark.functions import seq1

# Come up with a name for our new table
desired_table_name = "DEMO_TABLE"

# Just in case we want to run this cell more than once
print(f"Dropping table {desired_table_name} if it exists")
session.sql(f"DROP TABLE IF EXISTS {desired_table_name}").show()

# The SP returns an Any object and not a DataFrame
rows_desired = 7
print(f'FIRST: Invoking the stored procedure using session.call("{desired_table_name}",{rows_desired},False)')
sp_return_value = session.call("CREATE_AND_POP_TABLE_SP",desired_table_name,rows_desired,False) 
print(f"The return is: {sp_return_value}")

print(f"\nShowing all {session.table(desired_table_name).count()} rows")
(session.table(desired_table_name)
     .show(session.table(desired_table_name).count())
)

rows_desired = 5
print(f'SECOND: Invoking the stored procedure using session.call("{desired_table_name}",{rows_desired},False)')
sp_return_value = session.call("CREATE_AND_POP_TABLE_SP",desired_table_name,rows_desired,False) 
print(f"The return is: {sp_return_value}")

print(f"\nShowing all {session.table(desired_table_name).count()} rows")
(session.table(desired_table_name)
     .show(session.table(desired_table_name).count())
)

rows_desired = 2
print(f'THIRD: Invoking the stored procedure using session.call("{desired_table_name}",{rows_desired},True)')
sp_return_value = session.call("CREATE_AND_POP_TABLE_SP",desired_table_name,rows_desired,True) 
print(f"The return is: {sp_return_value}")

print(f"\nShowing all {session.table(desired_table_name).count()} rows")
(session.table(desired_table_name)
     .show(session.table(desired_table_name).count())
)

Dropping table DEMO_TABLE if it exists
------------------------------------
|"status"                          |
------------------------------------
|DEMO_TABLE successfully dropped.  |
------------------------------------

FIRST: Invoking the stored procedure using session.call("DEMO_TABLE",7,False)
The return is: 7 rows appended to table DEMO_TABLE

Showing all 7 rows
---------------------------------------
|"SEQUENCE"  |"Uniform Values 50-80"  |
---------------------------------------
|0           |50                      |
|1           |69                      |
|2           |57                      |
|3           |76                      |
|4           |64                      |
|5           |52                      |
|6           |71                      |
---------------------------------------

SECOND: Invoking the stored procedure using session.call("DEMO_TABLE",5,False)
The return is: 5 rows appended to table DEMO_TABLE

Showing all 12 rows
----------------------------------

---

By describing the function we can see that the code was small enough to be part of the function declaration in the Cloud Services layer of our Snowflake account. Listing the contents of the stage confirms that **no** `.py` files were created and placed into our stage. 

In [15]:
print("Describing procedure CREATE_AND_POP_TABLE_SP(...)")
desc_procedure_df = session.sql("DESCRIBE PROCEDURE CREATE_AND_POP_TABLE_SP(VARCHAR,NUMBER,BOOLEAN)")
desc_procedure_df.show(30, 90)

Describing procedure CREATE_AND_POP_TABLE_SP(...)
-------------------------------------------------------------------------------------------------------------------
|"property"          |"value"                                                                                     |
-------------------------------------------------------------------------------------------------------------------
|signature           |(ARG1 VARCHAR, ARG2 NUMBER, ARG3 BOOLEAN)                                                   |
|returns             |VARCHAR(16777216)                                                                           |
|language            |PYTHON                                                                                      |
|null handling       |CALLED ON NULL INPUT                                                                        |
|volatility          |VOLATILE                                                                                    |
|execute as          |

---

Notice with the output above that that `body` of the stored procedure contains a function (`func`) that holds  `bytes.fromhex(....)`. This is the pickled version of our Python stored procedure. 


In [ ]:
from snowflake.snowpark.functions import col
body = (desc_procedure_df
     .filter(col('"property"')==lit("body"))
     .select('"value"')
     .collect()[0][0]
)
print(body)

---

Because our stored procedure was so small, the registration process did not create a `.py` file and place it in the specified stage. The body of the SP was small enough to reside in metadata within the cloud services layer of our Snowflake account.

In [ ]:
# Nothing in the stage. 
print("No .py file placed in the stage")
session.sql(f"LIST @{stage_name}").show(100)

<a id="table_procedures"></a>

### 3C. Table Procedures a.k.a. Tabular Procedures

With the examples we have seen above for developing stored procedures using Snowpark for Python, the return type for our procedures was always a simple data type like `StringType`, `IntegerType`, `DoubleType`, etc. 

With Snowpark for Python, a stored procedure can return a result set of rows with columns similarly to that of a User-Defined Table Function (UDTF). 

> **&#128221; Note:** See documentation for further details:
> - [Stored Procedures - Returning Tabular Data](https://docs.snowflake.com/en/sql-reference/stored-procedures-python#returning-tabular-data)

See the lesson titled [*Creating and Registering User-Defined Functions (UDFs) and User-Defined Table Functions (UDTFs)*](./06-Creating-and-Registering-UDFs.ipynb) for more on UDFs and UDTFs. 

<img src="../../images/UDxFvSproc.png" alt="UDF vs Sproc calls" width=85% />

**Note:** Java and Scala stored procedures can also return tabular results but this is a preview feature for all Snowflake accounts as of Snowpark for Python `1.12.1`. 

<a id="The_RETURNS_Clause_in_SQL_CREATE_PROCEDURE_Statements"></a>
#### 3D. The `RETURNS` Clause in SQL `CREATE PROCEDURE` Statements

When defining your table procedure using SQL, the `RETURNS` clause would be `TABLE(col_name col_type,col_name col_type,...)` or even `TABLE()`.  Unlike a UDTF, the names and datatypes of the columns of the result set **do not need to be specified**. This means the output of a tabular stored procedure could be dynamic and not have the same structure in all circumstances. This is very powerful. 

```sql
CREATE PROCEDURE MY_PROCEDURE(PARAM1 TYPE1)
    RETURNS TABLE(COL1 TYPE1, COL2, TYPE2, ...) -- or just TABLE()
    LANGUAGE PYTHON
    HANDLER = 'run'
    ...
    AS $$
    
def run(session, p1):
    ...
    return (a,b,...)

    $$;
```

In [ ]:
# Import needed types, objects, and functions
from snowflake.snowpark import Session, DataFrame
from snowflake.snowpark.types import StructType, StructField 
import math

# Declare our Python function that will act as our handler function
def num_stuff(session:Session, num:float) -> DataFrame:
    typ = str(type(num)).split("'")[1]
    list_of_tuples = (
           [
               # col1 operation       col2 results
             (f"Number ({typ})",      num)
            ,(f"math.sqrt({num})",    math.sqrt(num))
            ,(f"math.pow({num},2)",   math.pow(num,2))
            ,(f"math.pow({num},3)",   math.pow(num,3))
            ,(f"math.log({num})",     math.log(num))
            ,(f"math.log10({num})",   math.log10(num))
            ,(f"math.sin({num})",     math.sin(num))
            ,(f"math.cos({num})",     math.cos(num))
            ,(f"math.tan({num})",     math.tan(num))
           ]
         )    
    return session.create_dataframe(list_of_tuples,schema=["MATH_OPERATION","RESULT"])

print(f"Python function num_stuff created: {num_stuff}")

We will need a stage to hold the bytecode of our procedure if we want it to be permanent. 

In [ ]:
sp_stage_name = 'SPROC_HOLDER_STAGE'
(session.sql(f"CREATE STAGE IF NOT EXISTS {sp_stage_name} ")
     .show()
)

Now that we have the function and a stage in which to put it, we can register our procedure. We will give it a name for use in SQL statements. 

From the [docs](https://docs.snowflake.com/en/developer-guide/snowpark/reference/python/latest/api/snowflake.snowpark.stored_procedure.StoredProcedureRegistration.register):
> A name must be specified when `is_permanent` is `True`.

In [ ]:
# Import needed types, objects, and functions
from snowflake.snowpark.functions import sproc
from snowflake.snowpark.types import StringType, DoubleType

# Register the SPROC with a name for SQL purposes 
num_stuff_sproc_instance = (
    sproc(
         func = num_stuff       
        ,input_types=[DoubleType()]   
        ,packages = ["snowflake-snowpark-python"]
        ,name =  "NUM_STUFF"                 # Name used in a SQL CALL statement or session.call(...)
        ,is_permanent = True                 # Temp procedure
        ,stage_location= f"@{sp_stage_name}" # Where to store the bytecode of the sproc
        ,replace = True                      # In case we want to run this cell again
        ,session = session                   # Session to use to register this UDTF
        # For the arguments named imports and packages, see the docs
    )
)

print(f"Stored procedure num_stuff_sproc_instance created: {num_stuff_sproc_instance}")
print("Describing the procedure")
session.sproc.describe(num_stuff_sproc_instance).show(100)

show_procedures_df.show(20)

The "table procedure" can now be invoked and will return row data in the form of a `DataFrame`.

In [ ]:
session.sql("CALL NUM_STUFF(10)").show()

session.call("NUM_STUFF",49.0).show()

num_stuff_sproc_instance(15).show()

<a id="Using_Libraries"></a>

### 3E. Using Custom and Third Party Libraries

There are two ways to use a custom or third party library in your procedure logic.
1. Importing Packages Through a Snowflake Stage
    - Upload the library (`.py` or `.zip`) to a stage and then reference the library with the `IMPORTS` argument e.g. `imports = ('@~/my_lib.py', '@some_stage/3rd_party.zip)`.
1. Using Third-Party Packages from Anaconda
    - If the library is managed by Snowflake Anaconda repository, you can reference the library (and optionally the version) using a `PACKAGES` argument e.g. `packages = ['numpy','xgboost==1.5.1']`.
    
> **&#128221; Note:** See documentation for further details:
> - [Importing Packages Through a Snowflake Stage](https://docs.snowflake.com/en/developer-guide/udf/python/udf-python-packages#importing-packages-through-a-snowflake-stage)
> - [Using Third-Party Packages from Anaconda](https://docs.snowflake.com/en/developer-guide/udf/python/udf-python-packages#using-third-party-packages-from-anaconda)
    
**Note:** Wheel (`.whl`) files are not directly supported but you can import and use the [Snowflake Labs](https://github.com/Snowflake-Labs) extension `wheel_loader.py` [found here](https://github.com/Snowflake-Labs/snowpark-extensions-py/tree/main/extras/wheel_loader) to unpack a wheel file programmatically. You will need to upload both your `.whl` file and `wheel_loader.py` to a stage and then import them in the same manner as above.
    
You can see what packages (and versions) are available from the view `INFORMATION_SCHEMA.PACKAGES`. 

In [ ]:
from snowflake.snowpark.functions import col, lit
(session.table("INFORMATION_SCHEMA.PACKAGES")
     .filter(col("LANGUAGE").like(lit("python")) & col("PACKAGE_NAME").like(lit("numpy%")))
     .show(20)
)

---
Let's use *numpy* in a stored procedure.





In [ ]:
# Stored procedure handler function
def numpy_random_with_seed(sess:Session, seed:int=None) -> float:
    import numpy as np # numpy lib does not have to be installed on the client
    np.random.seed(seed)
    return np.random.rand()

from snowflake.snowpark.functions import sproc
from snowflake.snowpark.types import DoubleType, IntegerType
numpy_sp = sproc( 
        func = numpy_random_with_seed 
       ,name="NUMPY_RANDOM_WITH_SEED"
       ,return_type = DoubleType()
       ,input_types = [IntegerType()]
       ,is_permanent= False
       ,replace= True
       ,packages=["snowflake-snowpark-python","numpy"] # Tell the server-side env what packages you need
       ,session = session
)
print("Create stored procedure NUMPY_RANDOM_WITH_SEED")

ran_10 = session.call("NUMPY_RANDOM_WITH_SEED",10)
print(f"Result of NUMPY_RANDOM_WITH_SEED(10): {ran_10}")
ran_3  = session.call("NUMPY_RANDOM_WITH_SEED",3)
print(f"Result of NUMPY_RANDOM_WITH_SEED(3): {ran_3}")

---
As always, the `StoredProcedure` object created by `sproc(...)`  allows us to invoke the procedure programmatically without using `session.sql("CALL ....")` or `session.call(...)`. 

In [ ]:
ran_7 = numpy_sp(7)
print(f"Result of numpy_sp: {ran_7}")

---
<a id="AE7_cleanup"></a>

## 4. Clean up and close the Session

When finished with your exercise, clean up the demo objects you created in your Snowflake account and close your `Session` object.

*You needn't edit anything in the following cell. Just run it.*

In [ ]:
close_session_and_clean_up(get_lesson())

### &#10071; `Shut Down Kernel`
> After completing the activities in a notebook and before moving on to the next exercise, shut down the completed notebook by right-clicking on the notebook name and selecting `Shut Down Kernel`.